# Ejercicio 2: Descenso de Gradiente
**Matemáticas para Inteligencia Artificial**

Función objetivo en forma matricial:
$$f(x) = \frac{1}{2}x^{\top}Ax - b^{\top}x$$

Con los datos del ejercicio:
$$A = \begin{pmatrix} 1 & 0 & 0 \\ 0 & 3 & 0 \\ 0 & 0 & 5 \end{pmatrix}, \quad b = \begin{pmatrix} -3 \\ 6 \\ -10 \end{pmatrix}, \quad \gamma = 0.25, \quad x_0 = \begin{pmatrix} 1 \\ 0 \\ 1 \end{pmatrix}$$

En este notebook comprobaremos:
1. La función y su gradiente (forma matricial vs. analítica).
2. El punto estacionario y su clasificación con la matriz Hessiana.
3. La tabla de las primeras 3 iteraciones ($x_0, x_1, x_2, x_3$) para validar los cálculos a mano.
4. La ejecución completa hasta alcanzar el mínimo ($\|\nabla f(x_i)\| < 10^{-6}$).


## 1. Importación de librerías y datos del problema


In [ ]:
import numpy as np              # Para operaciones con vectores y matrices
import pandas as pd             # Para mostrar los resultados en forma de tabla
import matplotlib.pyplot as plt # Para graficar el descenso de la función

# Matriz A y vector b del ejercicio
A = np.array([[1.0, 0.0, 0.0],
              [0.0, 3.0, 0.0],
              [0.0, 0.0, 5.0]])

b = np.array([-3.0, 6.0, -10.0])

# Parámetros del descenso de gradiente
gamma = 0.25                          # Tasa de aprendizaje
x0 = np.array([1.0, 0.0, 1.0])        # Punto inicial
tolerancia = 1e-6                     # Criterio de paro: ||gradiente|| < 10^-6
max_iteraciones = 1000                # Límite de seguridad para evitar ciclos infinitos

print("Matriz A:\n", A)
print("Vector b:", b)
print("Tasa de aprendizaje (gamma):", gamma)
print("Punto inicial (x0):", x0)

## 2. Definición de la función, gradiente y comprobación analítica


In [ ]:
# Función en forma matricial: f(x) = (1/2) * x^T * A * x - b^T * x
def f_matricial(x):
    return 0.5 * (x.T @ A @ x) - (b.T @ x)

# Función desarrollada como ecuación (inciso 1):
# f(x1, x2, x3) = 0.5*x1^2 + 1.5*x2^2 + 2.5*x3^2 + 3*x1 - 6*x2 + 10*x3
def f_ecuacion(x):
    x1, x2, x3 = x[0], x[1], x[2]
    return 0.5*(x1**2) + 1.5*(x2**2) + 2.5*(x3**2) + 3*x1 - 6*x2 + 10*x3

# Gradiente en forma matricial (como vector columna A*x - b para actualizar x)
def grad_matricial(x):
    return A @ x - b

# Gradiente con derivadas parciales analíticas (inciso 2)
def grad_analitico(x):
    x1, x2, x3 = x[0], x[1], x[2]
    df_dx1 = x1 + 3
    df_dx2 = 3*x2 - 6
    df_dx3 = 5*x3 + 10
    return np.array([df_dx1, df_dx2, df_dx3])

# Comprobamos con un punto de prueba (por ejemplo x = [1, 2, -1]) que ambas formas coinciden
x_prueba = np.array([1.0, 2.0, -1.0])
print("Comprobación en x =", x_prueba)
print("f matricial:", f_matricial(x_prueba), "| f ecuación:", f_ecuacion(x_prueba))
print("Gradiente matricial:", grad_matricial(x_prueba), "| Gradiente analítico:", grad_analitico(x_prueba))

## 3. Punto estacionario y clasificación con segundas derivadas (Hessiana)


In [ ]:
# Inciso 3: Resolver gradiente = 0 -> A*x = b
x_estacionario = np.linalg.solve(A, b)
print("Punto estacionario exacto (x*):", x_estacionario)
print("Valor mínimo exacto f(x*):", f_matricial(x_estacionario))

# Inciso 4: La matriz Hessiana de esta función cuadrática es exactamente la matriz A
H = A
print("\nMatriz Hessiana H:\n", H)

# Verificamos que todas las derivadas cruzadas (fuera de la diagonal) sean 0
fuera_diagonal = H - np.diag(np.diag(H))
cruzadas_cero = np.all(fuera_diagonal == 0)
diagonal_positiva = np.all(np.diag(H) > 0)

print("¿Todas las derivadas cruzadas son 0?:", cruzadas_cero)
print("Segundas derivadas puras (diagonal):", np.diag(H))

# Si las derivadas cruzadas son 0 y toda la diagonal es positiva, es un mínimo
if cruzadas_cero and diagonal_positiva:
    print("Clasificación: El punto estacionario es un MÍNIMO global.")

## 4. Ejecución del algoritmo de Descenso de Gradiente


In [ ]:
# Inicializamos el punto actual con x0
x = x0.copy()

# Lista para guardar los datos de cada iteración
historial = []

for i in range(max_iteraciones):
    # Calculamos el gradiente, su norma y el valor de la función en el punto actual
    grad = grad_matricial(x)
    norma_grad = np.linalg.norm(grad)
    valor_f = f_matricial(x)
    
    # Guardamos los datos de la iteración actual
    historial.append({
        "i": i,
        "x1": x[0],
        "x2": x[1],
        "x3": x[2],
        "grad_x1": grad[0],
        "grad_x2": grad[1],
        "grad_x3": grad[2],
        "||grad||": norma_grad,
        "f(x_i)": valor_f
    })
    
    # Criterio de paro: si la norma del gradiente es menor a la tolerancia, terminamos
    if norma_grad < tolerancia:
        break
    
    # Regla de actualización: x_{i+1} = x_i - gamma * gradiente
    x = x - gamma * grad

# Convertimos el historial en un DataFrame de pandas para verlo como tabla
tabla = pd.DataFrame(historial)

## 5. Comprobación de la tabla a mano (iteraciones $x_0$ a $x_3$)


In [ ]:
# Mostramos solo las primeras 4 filas (i = 0, 1, 2, 3) para comparar con los cálculos a mano
tabla_a_mano = tabla.head(4)
tabla_a_mano

## 6. Reporte del mínimo encontrado en Python


In [ ]:
# Tomamos la última fila de la tabla, que corresponde al punto de convergencia
resultado = tabla.iloc[-1]

print("=== RESULTADO FINAL ===")
print(f"Iteraciones totales: {int(resultado['i'])}")
print(f"Mínimo encontrado x*: [{resultado['x1']:.6f}, {resultado['x2']:.6f}, {resultado['x3']:.6f}]")
print(f"Valor mínimo f(x*):   {resultado['f(x_i)']:.6f}")
print(f"Gradiente final:      [{resultado['grad_x1']:.2e}, {resultado['grad_x2']:.2e}, {resultado['grad_x3']:.2e}]")
print(f"Norma del gradiente:  {resultado['||grad||']:.2e}")

## 7. Gráfica de convergencia de $f(x_i)$


In [ ]:
# Graficamos cómo disminuye f(x_i) en cada iteración hasta llegar al mínimo
plt.figure(figsize=(8, 4))
plt.plot(tabla["i"], tabla["f(x_i)"], marker="o", linewidth=2, label="f(x_i)")
plt.axhline(f_matricial(x_estacionario), color="red", linestyle="--", label="Mínimo exacto (-20.5)")

plt.title("Convergencia del Descenso de Gradiente - Ejercicio 2")
plt.xlabel("Iteración (i)")
plt.ylabel("Valor de f(x_i)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()